In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as plt_colors
import time
import UEG_response as ur

In [ ]:
# Units
hbar = 1.0
aB = 1.0
m = 1.0
e = 1.0

# Tolerances
reltol = 1e-16
abstol = 1e-8
eta_log  = 1e-6
eta_sqrt = 1e-6
eta_pol = 1e-4
points_n = 5
tol_upper = 1e-8
dx = 1e-4
lower = 1e-6
limit = 50



In [ ]:
# Temperature scan.
# Conditions
rs = 3.23
thetas = np.logspace(-2, np.log10(500), 100)


# Input
z1 = 0.0
y1 = 1.0
z2 = 0.1
y2 = 1.5
csTheta = 0.4


classical_chi2 = np.zeros(shape=thetas.shape, dtype=complex)
quantum_chi2   = np.zeros(shape=thetas.shape, dtype=complex)
norm_quantum   = np.zeros(shape=thetas.shape)
norm_classical = np.zeros(shape=thetas.shape)
for i, theta in enumerate(thetas):
    # Normalisation
    qF = (9*np.pi/4)**(1/3) / (rs*aB)
    EF = hbar**2 * qF**2 / (2*m)
    beta = 1/(theta*EF)
    n = 3/(4*np.pi*rs**3)
    beta_eff = beta / np.sqrt(1 + (1/theta)**2 )

    norm_quantum[i]   = n*beta_eff**2
    norm_classical[i] = n*beta**2

    # Physical units
    k1     = y1 * qF
    omega1 = z1  / (beta*hbar)
    k2     = y2 * qF
    omega2 = z2  / (beta*hbar)

    classical_chi2[i] = ur.classical_ideal_quadratic_response(k1, omega1, k2, omega2, csTheta, n, beta, m, dc=dx, eta_pol=eta_pol, reltol=reltol, abstol=abstol)[0]

    # Physical units
    k1     = y1 * qF
    omega1 = z1 / (beta_eff*hbar)
    k2     = y2 * qF
    omega2 = z2 / (beta_eff*hbar)

    quantum_chi2[i] =  ur.ideal_quadratic_response(omega1, k1, omega2, k2, csTheta,
                                                    m, hbar, n=n, beta=beta, ms=2,
                                                    reltol=reltol, abstol=abstol, eta_pol=eta_pol, eta_sqrt=eta_sqrt, eta_log=eta_log, tol_upper=tol_upper,
                                                    dx=dx, points_n=points_n, force_output=True)[0]

# Normalisation
qF = (9*np.pi/4)**(1/3) / (rs*aB)
EF = hbar**2 * qF**2 / (2*m)
n = 3/(4*np.pi*rs**3)

# Physical units
k1     = y1 * qF
omega1 = z1 * EF / hbar
k2     = y2 * qF
omega2 = z2 * EF / hbar
ground_state_chi2_0 = ur.ground_state_ideal_quadratic_response(omega1, k1, omega2, k2, csTheta, m, hbar, n, ms=2) * np.ones(shape=thetas.shape)

norm_ground_state = (n/EF**2) * np.ones(shape=thetas.shape)


plt.plot(thetas, np.real(quantum_chi2)/norm_quantum,   '-k', label=r"Quantum ($\beta_x = \beta_{eff}$)")
plt.plot(thetas, np.imag(quantum_chi2)/norm_quantum,   '-r')

plt.xscale('log')
plt.yscale('log')

yrange = plt.ylim()

plt.plot(thetas, np.real(classical_chi2)/norm_classical, '--k', label=r"Classical ($\beta_x = \beta$)")
plt.plot(thetas, np.imag(classical_chi2)/norm_classical, '--r')

plt.plot(thetas, np.real(ground_state_chi2_0)/norm_ground_state, ':k', label=r"$T = 0$ ($\beta_x = E_F^{-1}$)")
plt.plot(thetas, np.imag(ground_state_chi2_0)/norm_ground_state, ':r')


plt.xlim([np.min(thetas), np.max(thetas)])
# plt.ylim(yrange)

plt.legend()

plt.xlabel(r"$\Theta$")
plt.ylabel(r"$\chi^{(2)}_{0}(\vec{k}_1, \omega_1, \vec{k}_2, \omega_2)$ [$n\beta_{x}^2$]")

plt.text(2e-2, 2e-1, r"$\cos\theta = %g$"%(csTheta))

plt.ylim([1e-2, 1e0])
# plt.savefig(f"figures/limits_temperature.jpg", dpi=400, bbox_inches='tight')


In [ ]:
# Large k-asymptotes
# Conditions
rs = 3.23
theta = 1.0


# Normalisation
qF = (9*np.pi/4)**(1/3) / (rs*aB)
EF = hbar**2 * qF**2 / (2*m)
beta = 1/(theta*EF)
n = 3/(4*np.pi*rs**3)
beta_eff = beta / np.sqrt(1 + (1/theta)**2 )

norm = n*beta**2

# Limit as k1 -> 0
omega1 = 0.0 / (beta_eff*hbar)
omega2 = 0.0 / (beta_eff*hbar)
k2 = 2.0 * qF
csTheta = 0.4

k1 = np.logspace(-1, np.log10(50), 100) * qF


chi0_2 =  ur.ideal_quadratic_response(omega1, k1, omega2, k2, csTheta,
                                        m, hbar, n=n, beta=beta, ms=2,
                                        reltol=reltol, abstol=abstol, eta_pol=eta_pol, eta_sqrt=eta_sqrt, eta_log=eta_log, tol_upper=tol_upper,
                                        dx=dx, points_n=points_n, force_output=True)

E0 = hbar**2*k1**2/(2*m)
chi0_2_approx = - ur.ideal_linear_response(omega2, k2, m, hbar, n, beta, ms=2, 
                                           reltol=reltol, abstol=abstol, 
                                           eta_log=eta_log, tol_upper=tol_upper, points_n=points_n, force_output=True) / E0

chi0_2_0 =  ur.ideal_quadratic_response(omega1, 0.0, omega2, k2, csTheta,
                                        m, hbar, n=n, beta=beta, ms=2,
                                        reltol=reltol, abstol=abstol, eta_pol=eta_pol, eta_sqrt=eta_sqrt, eta_log=eta_log, tol_upper=tol_upper,
                                        dx=dx, points_n=points_n, force_output=True) * np.ones(shape=k1.shape)


plt.plot(k1/qF, np.real(chi0_2)/norm, '-k', label=r'$|\vec{k}_2| = %.1f\,q_F$'%(k2/qF))
plt.plot(k1/qF, np.real(chi0_2_approx)/norm, '--k')
plt.plot(k1/qF, np.real(chi0_2_0)/norm, ':k')

# Limit as k1 -> 0 and k2 -> 0
omega1 = 0.0 / (beta_eff*hbar)
omega2 = 0.0 / (beta_eff*hbar)
csTheta = 0.4
s = 1.3


k2 = s * k1


chi0_2 =  ur.ideal_quadratic_response(omega1, k1, omega2, k2, csTheta,
                                        m, hbar, n=n, beta=beta, ms=2,
                                        reltol=reltol, abstol=abstol, eta_pol=eta_pol, eta_sqrt=eta_sqrt, eta_log=eta_log, tol_upper=tol_upper,
                                        dx=dx, points_n=points_n, force_output=True)

E1 = hbar**2*k1**2/(2*m)
E2 = hbar**2*k2**2/(2*m)
E12 = hbar**2*(k1**2 + 2*csTheta*k1*k2 + k2**2)/(2*m)
chi0_2_approx = n * (E1 + E2 + E12)/(E1 * E2 * E12)

chi0_2_0 =  ur.ideal_quadratic_response(omega1, 0.0, omega2, 0.0, csTheta,
                                        m, hbar, n=n, beta=beta, ms=2,
                                        reltol=reltol, abstol=abstol, eta_pol=eta_pol, eta_sqrt=eta_sqrt, eta_log=eta_log, tol_upper=tol_upper,
                                        dx=dx, points_n=points_n, force_output=True) * np.ones(shape=k1.shape)


plt.plot(k1/qF, np.real(chi0_2)/norm, '-r', label=r'$|\vec{k}_2| = %.2f\,|\vec{k}_1|$'%(s))
plt.plot(k1/qF, np.real(chi0_2_approx)/norm, '--r')
plt.plot(k1/qF, np.real(chi0_2_0)/norm, ':r')

plt.xscale('log')
plt.yscale('log')

plt.xlabel(r"$|\vec{k}_1|/q_F$")
plt.ylabel(r"$\chi_0^{(2)}(\vec{k}_1, \vec{k}_2)$ [$n\beta_{eff}^2$]")
plt.legend()
plt.text(1.5e-1, 1e-3, r"$\cos\theta = %.1f$"%(csTheta))

plt.ylim([1e-4, 1e0])
plt.xlim([np.min(k1/qF), 20])

# plt.savefig(f"figures/limits_k.jpg", dpi=400, bbox_inches='tight')


In [ ]:
# Generlised plasma dispersion function.
a = np.linspace(0.0, 5.0, 50)
b = np.linspace(-5.0, 5.0, 51)

cs = [0.1, 1.0, 5.0]

sgn_c = 1.0

m = 0
ns = [1, 2, 3]

dc     = 1e-4
eta_pol= 1e-5
reltol = 1e-9
abstol = 1e-8


tmp = np.logspace(-4, 2.5, 14)
levels = np.concatenate( (-np.flip(tmp), np.array([0.0]), tmp) )
[A, B] = np.meshgrid(a, b)

levels_show = levels[1::2]
levels_show_format = []
for l in levels_show:
    if (l > 0):
        levels_show_format.append(r"$\;\;\;10^{%d}$"%(np.log10(np.abs(l))))
    else:
        levels_show_format.append(r"$-10^{%d}$"%(np.log10(np.abs(l))))


fig, axs = plt.subplots(3, 3, sharey=True, sharex=True)

plot_real = False


for i, c in enumerate(cs):
    for j, n in enumerate(ns):
        print(f"i = %d, j = %d"%(i,j))

        Y_flat = ur.generlized_plasma_dispersion_function_m_n(A.flatten(), B.flatten(), c, sgn_c, m, n, 
                                                           dc=dc, eta_pol=eta_pol, reltol=reltol, abstol=abstol)
        Y = np.reshape(Y_flat, shape=(len(b),len(a)))

        if (plot_real):
            Y_plot = np.real(Y)
        else:
            Y_plot = np.imag(Y)

        cmap = plt_colors.SymLogNorm(linthresh=1e-4, linscale=0.30, vmin=np.min(levels), vmax=np.max(levels), base=10)
        pp = axs[i,j].contourf(A, B, Y_plot, levels=levels, norm=cmap, cmap='seismic', alpha=0.8)
        cmap2 = plt_colors.LinearSegmentedColormap.from_list('mycmap', ['black', 'black', 'black'])
        pp2 = axs[i,j].contour( A, B, Y_plot, levels=levels, cmap=cmap2)

        if ( i == 0 ):
            # Plot title
            if (plot_real):
                axs[i,j].set_title(r"Re$Y_{0,%d}$"%(n))
            else:
                axs[i,j].set_title(r"Im$Y_{0,%d}$"%(n))
        if (i == 1 and j == 0):
            axs[i,j].set_ylabel(r"$b$")
        if (i == 2 and j == 1):
            axs[i,j].set_xlabel(r"$a$")


cbar = plt.colorbar(pp, format="%.1e", ticks=levels_show, ax=axs.ravel().tolist())
cbar.set_ticklabels(levels_show_format)
cbar.add_lines(pp2)

axs[0,0].text(0,5,r"(b)", color='w', verticalalignment='top')

# plt.savefig(f"figures/Y_comp_imag.jpg", dpi=400, bbox_inches='tight')


In [ ]:
def _norm_single(v):
  return np.sqrt( np.sum(v*v) )

def _Layden_ideal_quadratic_response(k1_vec, omega_1, k2_vec, omega_2, n, beta, m, dc=1e-4, eta_pol=1e-4, reltol=1e-6, abstol=1e-8):
    k_vec = k1_vec + k2_vec
    omega = omega_1 + omega_2

    vth = 1.0/np.sqrt(m*beta)


    k_cross_k1 = _norm_single(np.cross(k_vec, k1_vec))
    norm_k1    = _norm_single(k1_vec)
    norm_k2    = _norm_single(k2_vec)
    norm_k     = _norm_single(k_vec)

    # First M-term
    a  = omega   * norm_k / (k_cross_k1 * np.sqrt(2) * vth)
    a1 = omega_1 * norm_k / (k_cross_k1 * np.sqrt(2) * vth)
    a2 = omega_2 * norm_k / (k_cross_k1 * np.sqrt(2) * vth)

    b  = -np.dot(k_vec, k_vec)  / k_cross_k1
    b1 = -np.dot(k_vec, k1_vec) / k_cross_k1
    b2 = -np.dot(k_vec, k2_vec) / k_cross_k1

    s  = omega / (_norm_single( k_vec ) * np.sqrt(2) * vth)
    r = -(a1 + a2)/(b1 + b2)

    first = np.dot(k1_vec, k2_vec) * (ur.generlized_plasma_dispersion_function_m_n(a1, b1, s, 1.0, 0, 3, dc=dc, eta_pol=eta_pol, reltol=reltol, abstol=abstol)
                                    + ur.generlized_plasma_dispersion_function_m_n(a2, b2, s, 1.0, 0, 3, dc=dc, eta_pol=eta_pol, reltol=reltol, abstol=abstol) )

    # Second M-term
    a1t = omega   * norm_k1 / (k_cross_k1 * np.sqrt(2) * vth)
    a2t = omega_2 * norm_k1 / (k_cross_k1 * np.sqrt(2) * vth)
    b1t = b1
    b2t = -np.dot(k1_vec, k2_vec) / k_cross_k1

    s1  = omega_1 / (_norm_single( k1_vec ) * np.sqrt(2) * vth)
    rt = -(a1t + a2t)/(b1t + b2t)

    sgn_rt = 1.0

    tmp = ( ur.generlized_plasma_dispersion_function_m_n(a1t, b1t, rt, sgn_rt, 0, 1, dc=dc, eta_pol=eta_pol, reltol=reltol, abstol=abstol) 
          + ur.generlized_plasma_dispersion_function_m_n(a2t, b2t, rt, sgn_rt, 0, 1, dc=dc, eta_pol=eta_pol, reltol=reltol, abstol=abstol)
          - ur.generlized_plasma_dispersion_function_m_n(a1t, b1t, s1, 1.0, 0, 1, dc=dc, eta_pol=eta_pol, reltol=reltol, abstol=abstol)
          - ur.generlized_plasma_dispersion_function_m_n(a2t, b2t, s1, 1.0, 0, 1, dc=dc, eta_pol=eta_pol, reltol=reltol, abstol=abstol)
        ) / (rt - s1)

    tmp2 = ( tmp - ur.generlized_plasma_dispersion_function_m_n(a1t, b1t, s1, 1.0, 0, 2, dc=dc, eta_pol=eta_pol, reltol=reltol, abstol=abstol)
                 - ur.generlized_plasma_dispersion_function_m_n(a2t, b2t, s1, 1.0, 0, 2, dc=dc, eta_pol=eta_pol, reltol=reltol, abstol=abstol)
           ) / (rt - s1)

    second = np.dot(k_vec, k2_vec) * norm_k1**2 * tmp2 / (np.dot(k_vec,k1_vec) + np.dot(k1_vec,k2_vec))

    # Third M-term
    a1b = omega_1 * norm_k2 / (k_cross_k1 * np.sqrt(2) * vth)
    a2b = omega   * norm_k2 / (k_cross_k1 * np.sqrt(2) * vth)
    b1b = b2t
    b2b = b2

    s2  = omega_2 / (_norm_single( k2_vec ) * np.sqrt(2) * vth)
    rb = -(a1b + a2b)/(b1b + b2b)

    sgn_rb = 1.0

    tmp = ( ur.generlized_plasma_dispersion_function_m_n(a1b, b1b, rb, sgn_rb, 0, 1, dc=dc, eta_pol=eta_pol, reltol=reltol, abstol=abstol) 
          + ur.generlized_plasma_dispersion_function_m_n(a2b, b2b, rb, sgn_rb, 0, 1, dc=dc, eta_pol=eta_pol, reltol=reltol, abstol=abstol)
          - ur.generlized_plasma_dispersion_function_m_n(a1b, b1b, s2, 1.0, 0, 1, dc=dc, eta_pol=eta_pol, reltol=reltol, abstol=abstol)
          - ur.generlized_plasma_dispersion_function_m_n(a2b, b2b, s2, 1.0, 0, 1, dc=dc, eta_pol=eta_pol, reltol=reltol, abstol=abstol)
        ) / (rb - s2)

    tmp2 = ( tmp - ur.generlized_plasma_dispersion_function_m_n(a1b, b1b, s2, 1.0, 0, 2, dc=dc, eta_pol=eta_pol, reltol=reltol, abstol=abstol)
                 - ur.generlized_plasma_dispersion_function_m_n(a2b, b2b, s2, 1.0, 0, 2, dc=dc, eta_pol=eta_pol, reltol=reltol, abstol=abstol)
           ) / (rb - s2)

    third = np.dot(k_vec, k1_vec) * norm_k2**2 * tmp2 / (np.dot(k_vec,k2_vec) + np.dot(k1_vec,k2_vec))


    # Full expression for response function
    return (n / m**2) * (first + second + third) / (4 * vth**4 * k_cross_k1)

In [ ]:
# Conditions
rs = 3.23
thetas = np.logspace(-1, np.log10(200), 30)



dc        = 1e-4
eta_pol   = 1e-5
reltol    = 1e-9
abstol    = 1e-8
eta_sqrt  = 1e-6
eta_log   = 1e-6
tol_upper = 1e-8

csThetas = np.array([0.1, 0.3, 0.7])

fig, axs = plt.subplots(1, len(csThetas), sharex=True, sharey=True, figsize=(len(csThetas)*6.4, 1*4.8), )

for j, csTheta in enumerate(csThetas): 

    Layden_chi2    = np.zeros(shape=thetas.shape, dtype=complex)
    classical_chi2 = np.zeros(shape=thetas.shape, dtype=complex)
    quantum_chi2   = np.zeros(shape=thetas.shape, dtype=complex)
    norm           = np.zeros(shape=thetas.shape)
    for i, theta in enumerate(thetas):
        inv_theta = 1/theta

        # Units
        hbar = 1.0
        aB = 1.0
        m = 1.0
        e = 1.0
        eps0 = 1/(4*np.pi)

        # Normalisation
        qF = (9*np.pi/4)**(1/3) / (rs*aB)
        EF = hbar**2 * qF**2 / (2*m)
        beta = 1/(theta*EF)
        n = 3/(4*np.pi*rs**3)
        omega_p = np.sqrt(n*e**2/(m*eps0))
        beta_eff = beta / np.sqrt(1 + (1/theta)**2 )
        norm[i] = n*beta_eff**2
        

        # Inputs
        k1 = 1.0 * qF
        k1_vec = np.array([0,  0, 1]) * k1
        omega_1 = 0.0 * EF / hbar

        k2 = 0.5 * qF
        k2_vec = np.array([0, np.sqrt(1 - csTheta**2), csTheta]) * k2
        omega_2 = 0.01 * EF / hbar

        Layden_chi2[i] = 0.5 * _Layden_ideal_quadratic_response(k1_vec, omega_1, k2_vec, omega_2, n, beta, m, dc=dc, eta_pol=eta_pol, reltol=reltol, abstol=abstol)[0]

        classical_chi2[i] = ur.classical_ideal_quadratic_response(k1, omega_1, k2, omega_2, csTheta, n, beta, m, dc=dc, eta_pol=eta_pol, reltol=reltol, abstol=abstol)[0]


        quantum_chi2[i] =  ur.ideal_quadratic_response(omega_1, k1, omega_2, k2, csTheta,
                                     m, hbar, n=n, beta=beta, ms=2,
                                     reltol=reltol, abstol=abstol, eta_pol=eta_pol, eta_sqrt=eta_sqrt, eta_log=eta_log, tol_upper=tol_upper,
                                     dx=dc, points_n=5, force_output=True)[0]
    
    axs[j].plot(thetas, np.real(quantum_chi2)/norm,   '-k',  label=r"Quantum (real)")
    axs[j].plot(thetas, np.imag(quantum_chi2)/norm,   '-r',  label=r"Quantum (imag)")

    axs[j].plot(thetas, np.real(classical_chi2)/norm, '--k', label=r"Classical (real)")
    axs[j].plot(thetas, np.imag(classical_chi2)/norm, '--r', label=r"Classical (imag)")

    axs[j].plot(thetas, np.real(Layden_chi2)/norm, '-.k', label=r"$\frac{1}{2} \times $ Layden et. al (real)")
    axs[j].plot(thetas, np.imag(Layden_chi2)/norm, '-.r', label=r"$\frac{1}{2} \times $ Layden et. al (imag)")

    axs[j].set_xlabel(r"$\Theta$")
    axs[j].text(1e1, 5e-2, r"$\cos\theta = %g$"%(csTheta))

plt.xscale('log')
plt.yscale('log')


plt.xlim([np.min(thetas), np.max(thetas)])

plt.legend(ncol=3, loc="upper right")

axs[0].set_ylabel(r"$\chi^{(2)}_{0}(\vec{k}_1, \omega_1, \vec{k}_2, \omega_2)$ [$n\beta_{eff}^2$]")


# np.real(classical_chi2[-1])/np.real(quantum_chi2[-1]), np.imag(classical_chi2[-1])/np.imag(quantum_chi2[-1]), np.real(Layden_chi2[-1])/np.real(quantum_chi2[-1]), np.imag(Layden_chi2[-1])/np.imag(quantum_chi2[-1])

fig.subplots_adjust(wspace=0.0)

# plt.savefig(f"figures/classical_comparison_combined.jpg", dpi=400, bbox_inches='tight')
